---
title: "Chapter 33: Predicting household spend"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/05-classical-ml-projects/33-spend-regression.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/05-classical-ml-projects/33-spend-regression.ipynb)

In November 2019, a single-person household in Alabama, reporting an income between $25,000 and $49,999, bought a 65-inch Samsung 4K television, a PlayStation 4 Pro, a Microsoft Surface tablet, an Apple Watch, a GoPro camera bundle, a trampoline, and a kayak. Fifty-eight items, spread across four days. The total came to $11,709.16.

For the three months before that, the same household had spent an average of $49.47 a month. Across the roughly four years this account appears in the data, its lifetime total is $16,237.77. One month accounts for 72 percent of everything it has ever bought on the platform.

Nothing in this household's survey answers, age bracket, income, education, state, explains it. There's no flag for it and no obvious trigger. It's one row among 1.85 million, sitting in the data whether or not anyone thinks to look for it.

Chapters 28 through 32 gave you the machinery for exactly this kind of question: how to turn a vague business problem into a five-component specification, how the nine-stage workflow moves data through a model and into production, how scikit-learn's Transformer and Predictor interfaces work, how a Pipeline strings them together without leaking information across a split. You used all of it on hotel booking data that was already clean enough to teach with.

Here, you use it on a dataset nobody cleaned for teaching: five years of real Amazon purchase histories from 5,027 U.S. households, linked to a demographic and lifestyle survey each household completed once. The household above is what that data looks like up close. It's also the first of seven case studies built the same way, one real dataset and one real business question each, so the habit you build in this chapter (reason through the ask, prototype in the notebook, promote what works into a tested module) is the one the other six will follow.

## 1. Problem framing

Here's the request, exactly as it would actually arrive. You're the data scientist on a retention team at a company sitting on data like this, purchase histories linked to a demographic survey, for a panel of customers. Your manager stops by your desk and says:

"We want to know which customers are worth investing more in."

That's the whole brief. No target variable, no metric, no mention of a model. This is what a real ask sounds like, and your job right now is to figure out what it actually means, not to open a notebook.

### What does "worth investing in" even mean?

Before you touch a single file, you need to pull that sentence apart. It's collapsing at least four different problems into one, and they are not interchangeable:

- Customers who already spend the most. A ranking of the past, nothing predictive about it.
- Customers likely to spend more going forward. A forecast.
- Customers likely to leave if nobody intervenes. A churn problem.
- Customers who look like existing high spenders but haven't been identified yet. A segmentation problem.

Each one is a legitimate data science problem. Each one produces a different target variable, a different model, and a different action the business takes on your output. Guess wrong here and you could spend six months building the right system for the wrong question.

You go back and ask a follow-up question. It turns out "worth investing in" means deciding who gets priority for a loyalty program with a fixed, limited budget. The team wants customers ranked by how much they're likely to spend going forward, using the demographic and behavioural data already on file, before that spending happens. Not a scoreboard of who has already spent the most.

That answer resolves the first fork: this is a forecast of future value, not a description of past value, which rules out simply sorting by historical total. It also starts pinning down [Chapter 30's five-component spec](../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem), one piece at a time, following wherever your own follow-up questions lead next:

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: write the spec card before reading further</span><br><br>
Using [Chapter 30's five-component spec](../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem) (target variable, task type and paradigm, success metric, acceptable threshold, data specification), write your own answer for this problem before reading the discussion below. Compare where you landed once you get there. If your target variable differs, that's fine, it's the kind of decision Chapter 30's Pro Tip already warned about: agree it with the stakeholder who will actually use the model, because a model optimised toward the wrong objective will hit that objective and still miss the point.
</div>

**Target variable.** Not last month's spend, what you're predicting is next year's. You still need a window, a month, a year, a lifetime span all answer the question differently, and the right one depends on how often the loyalty program actually reallocates its budget. If the business revisits the list annually, that settles it: a household's spend over the coming twelve months.

**Task type and paradigm.** You're predicting a number from labelled historical examples where the outcome is already known. That's regression, supervised.

**Success metric.** The team isn't going to read predictions one row at a time, they're going to act on a ranked list. So the metric that matters is whether your model orders households correctly relative to each other, and whether the top decile it flags genuinely outspends everyone else. Mean absolute error is still worth tracking, mostly as a sanity check that the numbers themselves aren't nonsense.

**Acceptable threshold.** A model that can't beat "assume everyone spends the average" isn't worth shipping. Here's the real bar: households you rank in the top decile need to outspend a randomly chosen household by enough that the loyalty program's cost per member actually pays for itself, a number the business has to hand you, you can't derive it from the data alone.

**Data specification.** One demographic and lifestyle survey per household, answered once, and multi-year purchase history at the individual-transaction level, for 5,027 households.

::: {.callout-note collapse="true" icon=false}
## The resolved spec card

| Component | Specification |
|---|---|
| Target variable | Household's total spend over the coming 12 months |
| Task type and paradigm | Regression, supervised |
| Success metric | Rank quality (does the top decile of predicted spend actually outspend the rest), MAE as a secondary check |
| Acceptable threshold | Top-decile households must outspend a randomly chosen household by a margin the loyalty program's budget can justify |
| Data specification | One-time demographic and lifestyle survey plus multi-year, transaction-level purchase history, 5,027 U.S. households |
:::

That's the spec, and it took five follow-up questions to get there, not a target variable declared in the first paragraph. Every one of those questions is something a real stakeholder conversation would actually raise, and you'll ask versions of them again in every case study that follows.

Next: turning this into something durable enough that the rest of the project can't quietly drift from it, a Pydantic object, not a paragraph you could reinterpret three sections from now.

### Making the decision durable

Right now the spec card is a table in a notebook. Nothing stops the next person, or you, in three weeks, from reading "predicted spend" and quietly building a model for last month's spend instead, because nothing enforces which one was actually decided. A Pydantic object does. Once `settings.spec.target_variable` is a real attribute, every later section imports it instead of re-reading a paragraph and hoping it remembers correctly.

This is also where the practical decisions live, the ones that don't belong in a business spec but still need a fixed answer somewhere: which Harvard Dataverse files this pulls from, where they get cached, and which survey questions never make it into a feature, no matter how predictive they turn out to be.

In [ ]:
from pathlib import Path

from pydantic import BaseModel
from pydantic_settings import BaseSettings, SettingsConfigDict


class SpecCard(BaseModel):
    """The five-component spec resolved above, kept as data, not prose."""

    target_variable: str = "Household's total spend over the coming 12 months"
    task_type: str = "Regression, supervised"
    success_metric: str = "Rank quality (top-decile lift over random), MAE as a secondary check"
    acceptable_threshold: str = (
        "Top-decile households must outspend a random household by a margin the loyalty program's budget can justify"
    )
    data_specification: str = (
        "One-time demographic and lifestyle survey plus multi-year, "
        "transaction-level purchase history, 5,027 U.S. households"
    )


class SpendRegressionSettings(BaseSettings):
    """Paths, source data, and policy decisions for this case study."""

    model_config = SettingsConfigDict(env_file=".env", env_prefix="SPEND_", case_sensitive=False)

    spec: SpecCard = SpecCard()

    data_dir: Path = Path("data")
    dataverse_base_url: str = "https://dataverse.harvard.edu/api/access/datafile"
    purchases_file_id: int = 7616235  # amazon-purchases.csv, ~298.5 MB
    survey_file_id: int = 7616231  # survey.csv, tab-separated despite the .csv extension

    random_seed: int = 42

    excluded_survey_columns: list[str] = [
        "Q-substance-use-cigarettes",
        "Q-substance-use-marijuana",
        "Q-substance-use-alcohol",
        "Q-personal-diabetes",
        "Q-personal-wheelchair",
        "Q-sexual-orientation",
    ]


settings = SpendRegressionSettings()
settings.spec.target_variable

In [ ]:
from great_tables import GT, md as gt_md
import pandas as pd

from ark.plot.gt_style import themed_gt

LABELS = {
    "target_variable": "Target variable",
    "task_type": "Task type and paradigm",
    "success_metric": "Success metric",
    "acceptable_threshold": "Acceptable threshold",
    "data_specification": "Data specification",
}
spec_df = pd.DataFrame([{"Component": LABELS[k], "Specification": v} for k, v in settings.spec.model_dump().items()])
themed_gt(GT(spec_df), n_rows=len(spec_df)).tab_header(
    title=gt_md("**The spec card, as code**"),
    subtitle="Generated from settings.spec, not retyped from the table above",
)

Same five rows as the resolved spec card earlier in this chapter. The difference is where they came from: that one was a markdown table anyone could edit without noticing, this one is read straight off the object the rest of the pipeline will actually import.

This is what "promoted" means in practice. The two classes above are not staying in this notebook, they live at `ark/case_studies/spend_regression/config.py`, word for word. Proving that means checking the values survive the move, then using the real, imported class for everything from here on, not the copy defined above:

In [ ]:
before_spec = settings.spec.model_dump()
before_exclusions = settings.excluded_survey_columns

from ark.case_studies.spend_regression.config import SpendRegressionSettings

settings = SpendRegressionSettings()
settings.spec.model_dump() == before_spec and settings.excluded_survey_columns == before_exclusions

No test file for this one. Pydantic already validates the structure on construction, and there's no logic here yet worth a regression test, just decisions. That changes at the next step: downloading the actual data is where something can genuinely go wrong (a bad URL, a schema drift, a file Harvard renames), and where this chapter writes its first real test.

## 2. Data collection

The data specification says a one-time survey plus multi-year purchase history, for 5,027 households, hosted on Harvard Dataverse. Downloading a file isn't a data science problem, it's infrastructure, so you're not building it by hand in this notebook. It's a utility: written once, tested, imported everywhere it's needed, the same way you'd reach for a library function instead of re-deriving it.

In [ ]:
from ark.case_studies.spend_regression.data import load_survey

Behind that import is `download_dataverse_file`, which handles the parts that have nothing to do with this dataset specifically and everything to do with downloading files correctly:

- **Idempotent.** You won't wait through a 300 MB download twice, once a file is on disk, this skips the network entirely.
- **Streamed in chunks.** Not read into memory in one call, so a large file doesn't need to fit in RAM before it's even written to disk.
- **HTTPS-only, on purpose.** It refuses any URL that isn't `https://` before opening a connection. Dataverse would never hand back anything else, but you shouldn't write a function that fetches arbitrary URLs and quietly trusts whatever string it's given.
- **A plain header fixes a confusing error.** Dataverse's server rejects Python's default `urllib` user agent with a 403, even though the same URL works fine in `curl` or a browser. You send a normal-looking header instead, one line, and the "broken API" turns out to be a request that just looked wrong.

None of that is specific to spend prediction. You're looking at code worth writing once, testing, and never thinking about again. That's why it gets its own test, next.

This is also the repo's first real test file, `tests/case_studies/test_spend_regression.py`, three tests that never touch the network: one confirms the cache short-circuit works without a real download, one confirms the HTTPS guard rejects a bad scheme, one confirms the tab-separator handling on a small fixture file. Run it yourself here rather than take it on faith:

In [ ]:
!uv run pytest ../../tests/case_studies/test_spend_regression.py -v --override-ini=addopts=""

### Loading the survey

Before loading it: `survey.csv` is tab-separated. Nothing about the extension warns you, and `pd.read_csv` with default settings would silently parse it as one giant column instead of raising an error, a worse failure than a crash because it looks like it worked. `load_survey` already accounts for this.

In [ ]:
survey_df = load_survey(settings)
themed_gt(GT(survey_df.head(5)), n_rows=5).tab_header(
    title=gt_md("**First 5 survey responses**"),
    subtitle=f"{survey_df.shape[1]} columns per household",
)

Now the exclusion list in `config.py` makes sense to look at directly instead of taking on faith. Scroll through the columns above and you'll find `Q-demos-age`, `Q-demos-income`, `Q-demos-education` sitting a few columns away from `Q-substance-use-cigarettes`, `Q-personal-diabetes`, `Q-personal-wheelchair`, and `Q-sexual-orientation`. One survey, one consent form, questions about spending habits next to questions that have nothing to do with them.

None of the second group predicts spend meaningfully better than the first, and even if one of them did, using it would mean a model that quietly scores people differently by health status or sexual orientation, without anyone deciding that on purpose. `excluded_survey_columns` is what stops that from happening by default. It's not a privacy footnote, it's a feature-selection decision, made once, here, instead of left for whoever writes `features.py` to notice or miss.